In [9]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

## Simple LLM

In [11]:
## simple_llm_app.py

# initialize the model
model = ChatOpenAI(model="gpt-4o-mini")

# create a prompt template
prompt=PromptTemplate(template="Suggest a catchy blog title about {topic}",
                        input_variables=["topic"])

# define the input
topic=input("Enter the topic you want a blog title for:")
print("Topic is:", topic)

# format the promp manually using prompt template

# formatted_prompt=prompt.format(topic=topic) # older code -- IGNORE ---
formatted_prompt=prompt.invoke({"topic":topic})

# call the llm directly
# blog_title = model.predict(formatted_prompt)  # older code -- IGNORE ---
blog_title=model.invoke(formatted_prompt)

# print the response
print("Generated Blog Title:",blog_title.content)

Topic is: GenAI
Generated Blog Title: "GenAI Unleashed: Transforming Ideas into Reality!"


## PDF Reader (RAG Architecture)

In [ ]:
##pdf_reader.py

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS

load_dotenv()


# load the document
loader=TextLoader("state_of_the_union.txt", encoding="utf8")
documents=loader.load()

# split the text into smaller chunks
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs=text_splitter.split_documents(documents)

# convert text embeddings and store in FAISS
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore=FAISS.from_documents(docs, embeddings)

# create a retriever (fetches relevant documents)
retriever=vectorstore.as_retriever()

# Manually retrieve relevant documents
query="What are the key takeaways from the document?"
# retrieved_docs=retriever.get_relevant_documents(query) # older code -- IGNORE ---
retrieved_docs=retriever.invoke(query)

# combine the retrieved documents into a single string
context="\n".join([doc.page_content for doc in retrieved_docs])

# initialize the model
model=ChatOpenAI(model="gpt-4")

# manually pass retrieved text to llm
prompt=f"Based on the following context, answer the question:\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"
response=model.invoke(prompt)

# print the answer
print("Answer:", response.content)

Answer: The key takeaways from the document include the need for immigration reform that provides a pathway to citizenship for Dreamers, temporary status holders, farm and essential workers. Law revisions are advocated for business growth and faster family reunions. Advocacy for ARPA-H, a project aimed at medical breakthroughs in diseases like cancer, Alzheimer's and diabetes is expressed. The speaker highlights the strength of the Union and its citizens, emphasizing opportunities and growth. Finally, the document mentions efforts made to confront Putin's aggressions in alliance with other freedom-loving nations. The importance of truth and accountability is also stressed.


## LLM Chain

In [13]:
# llmchain.py
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

load_dotenv()


import os 
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# ♦ Load the LLM (GPT-3.5)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# ♦ Create a Prompt Template
prompt = PromptTemplate(
    input_variables=["topic"], # Defines what input is needed
    template="Suggest a catchy blog title about {topic}."
)

# ♦ Create an LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

# ♦ Run the chain with a specific topic
topic = input('Enter a topic: ')
output = chain.run(topic)

print("Generated Blog Title:", output)

Generated Blog Title: "Unleashing the Power of GenAI: A Journey into Artificial Intelligence"


## RetrievalQA

In [15]:
# pdf_reader.py

import os
# Set your OpenAI API key as an environment variable (recommended)
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_classic.document_loaders import TextLoader 
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter 
from langchain_classic.chains import RetrievalQA

# --- Stage 1: Load and Split Document ---

# ♦ Load the document
# NOTE: Ensure a file named 'state_of_the_union.txt' exists in the same directory 
# or change the path to a valid text file.
loader = TextLoader("state_of_the_union.txt", encoding="utf8") 
documents = loader.load()

# ♦ Split the text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# --- Stage 2: Embedding and Storage (Vector Store) ---

# ♦ Convert text into embeddings & store in FAISS
# This requires the OpenAI API key to be set
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())

# --- Stage 3: Retrieval ---

# ♦ Create a retriever (fetches relevant documents)
retriever = vectorstore.as_retriever()


# ♦ Initialize the LLM
llm = ChatOpenAI(model="gpt-4", temperature=0.7)

qa_chain = RetrievalQA.from_chain_type(llm=llm,retriever=retriever,chain_type="stuff")

query = "What are the key takeaways from the document?"
answer = qa_chain.run(query)
# ♦ Print the Answer
print("Query:", query)
print("Answer:", answer)

Query: What are the key takeaways from the document?
Answer: The key takeaways from the document are:

1. The speaker is expressing a deep belief in the strength, resilience, and potential of the nation, stating that it has overcome numerous challenges in the past and will continue to do so.
2. The speaker emphasizes the need for collective responsibility and resolve in the face of trials, asserting that the character and future of the nation are formed in such moments.
3. The speaker advocates for a focus on health issues, specifically addressing addiction and mental health, especially among children.
4. The speaker is optimistic about the future of the nation, stating that there is nothing beyond its capacity and every crisis has been transformed into an opportunity.
5. The speaker states that the State of the Union is strong and will continue to grow stronger, attributing this strength to the people of the nation.
6. The speaker calls for unity and collective action to meet and over

## Problem with Older Langchain

In [16]:
import random

# problems
class DummyLLM():
    def __init__(self):
        print("DummyLLM initialized")

    def predict(self, prompt):
        response_list=[
            "This is a dummy response.",
            "DummyLLM says hello!",
            "How can I assist you today?"
        ]
        return {'response': random.choice(response_list)}

llm=DummyLLM()
prompt="Tell me a joke."

response=llm.predict(prompt)
print("Response:", response['response']) 

DummyLLM initialized
Response: How can I assist you today?


In [17]:
class DummyPromptTemplate():    
    def __init__(self, template, input_variables):
        self.template=template
        self.input_variables=input_variables

    def format(self, **input_dict):
        return self.template.format(**input_dict)
    

template=DummyPromptTemplate(template="Tell me a {length}joke about {topic}.", 
                             input_variables=["length","topic"])


formatted_prompt=template.format(length="short",topic="computers")
print("Formatted Prompt:", formatted_prompt)

response=llm.predict(formatted_prompt)
print("Formatted Response:", response['response']) 

Formatted Prompt: Tell me a shortjoke about computers.
Formatted Response: This is a dummy response.


In [18]:
class DummyLLMChain():
    def __init__(self, llm, prompt):
        self.llm=llm
        self.prompt=prompt

    def run(self, input_dict):
        formatted_prompt=self.prompt.format(**input_dict)
        result=self.llm.predict(formatted_prompt)
        return result['response']
    

In [ ]:
template=DummyPromptTemplate(template="Tell me a {length}joke about {topic}.", 
                             input_variables=["length","topic"])

llm=DummyLLM()
chain=DummyLLMChain(llm=llm, prompt=template)
response=chain.run({"length":"long","topic":"AI"})
print("Chain Response:", response)


## Solution - Runnable

In [34]:
from abc import ABC, abstractmethod
import random

class Runnable(ABC):
    @abstractmethod
    def invoke(input_data):
        pass

In [35]:
import random

# problems
class DummyLLM(Runnable):
    def __init__(self):
        print("DummyLLM initialized")

    def invoke(self, input_data):
        response_list=[
            "This is a dummy response.",
            "DummyLLM says hello!",
            "How can I assist you today?"
        ]
        return {'response': random.choice(response_list)}
       

    def predict(self, prompt):
        response_list=[
            "This is a dummy response.",
            "DummyLLM says hello!",
            "How can I assist you today?"
        ]
        return {'message':'this method is going to be deprecated','response': random.choice(response_list)}

In [40]:
class DummyPromptTemplate(Runnable):    
    def __init__(self, template, input_variables):
        self.template=template
        self.input_variables=input_variables

    def invoke(self, input_data):
        return self.template.format(**input_data) # <-- unpack dict
    
    def format(self, input_dict):
        return self.template.format(**input_dict) # <-- unpack dict
    

In [52]:
class DummyStrOutputParser(Runnable):

    def __init__(self):
        pass

    # def parse(self, llm_output):
    #     return llm_output['response']
    
    def invoke(self, llm_output):
        return llm_output['response']

In [44]:
class RunnableConnect(Runnable):
    def __init__(self, runnables_list):
        self.runnables_list=runnables_list

    def invoke(self, input_data):
        
        for runnable in self.runnables_list:
            input_data=runnable.invoke(input_data)
        return input_data

In [45]:
llm=DummyLLM()
template=DummyPromptTemplate(template="Tell me a {length}joke about {topic}.", 
                             input_variables=["length","topic"])

parser=DummyStrOutputParser()

chain=RunnableConnect([template, llm, parser])

response=chain.invoke({"length":"short","topic":"computers"})
print(response)

DummyLLM initialized
How can I assist you today?


In [46]:
prompt_template1=DummyPromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

prompt_template2=DummyPromptTemplate(
    template='Explain the following joke {response}',
    input_variables=['response']
)

In [47]:
llm=DummyLLM()
parser=DummyStrOutputParser()

DummyLLM initialized


In [51]:
chain1=RunnableConnect([prompt_template1, llm])
chain2=RunnableConnect([prompt_template2, llm, parser])

final_chain=RunnableConnect([chain1, chain2])
final_chain.invoke({"topic":"cricket"})


'This is a dummy response.'